In [ ]:
import os
import pandas as pd
from datetime import datetime
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import re


# make sure to have dcm2niix installed
from utils.dcm2niix import *

!which dcm2niix


BERNER_DATEN_PATH='/nasdata/Berner_Daten/Other_Side_Bilddaten'
# read
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'

APP_DATA_PATH = '/home/bule/berner_daten_analyse/apps/app_manual_preselection'

In [ ]:
# Widgets
patient_dropdown = Dropdown(options=df['patient_id'].unique(), description='Patient:')
sequence_dropdown = Dropdown(description='Sequence:')
sequences_output = Output()
output = Output()

# Update sequence dropdown and display all entries for the selected patient
def update_sequences(*args):
    patient_id = patient_dropdown.value
    patient_data = df[df['patient_id'] == patient_id]
    sequence_dropdown.options = patient_data['study_id'].values

    with sequences_output:
        clear_output(wait=True)
        display(patient_data[['study_id', 'file_path', 'modality']])

# Display the selected modality visualization
def display_modality():
    clear_output(wait=True)
    patient_id = patient_dropdown.value
    study_id = sequence_dropdown.value

    if study_id is None:
        return

    row = df[(df['patient_id'] == patient_id) & (df['study_id'] == study_id)].iloc[0]
    file_path = row['file_path']
    modality = row['modality']

    # Load NIfTI file
    img = nib.load(file_path)
    data = img.get_fdata()

    def view_slice_dwi(slice_idx):
        clear_output(wait=True)
        plt.figure(figsize=(3, 3))
        plt.imshow(data[:, :, slice_idx, 1].squeeze(), cmap="gray", origin="lower")
        plt.title(f"Slice {slice_idx}")
        plt.axis("off")
        plt.show()

    def view_slice(slice_idx):
        clear_output(wait=True)
        plt.figure(figsize=(3, 3))
        plt.imshow(data[:, :, slice_idx].squeeze(), cmap="gray", origin="lower")
        plt.title(f"Slice {slice_idx}")
        plt.axis("off")
        plt.show()

    if modality == 'dwi':
        slider = IntSlider(min=0, max=data.shape[2] - 1, step=1, value=0)
        interactive_plot = interactive(view_slice_dwi, slice_idx=slider)
    elif modality == 't1':
        slider = IntSlider(min=0, max=data.shape[2] - 1, step=1, value=0)
        interactive_plot = interactive(view_slice, slice_idx=slider)

    display(interactive_plot)

# Link dropdowns and output
patient_dropdown.observe(update_sequences, names='value')
sequence_dropdown.observe(lambda change: display_modality(), names='value')

# Initial population of sequences
update_sequences()

# Layout
display(VBox([Label('Select Patient and Sequence:'), patient_dropdown, sequences_output, sequence_dropdown, output]))


### first try with manual selection neg / pos

In [ ]:
dwi_t1_all_df=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_all.csv"))
dwi_t1_all_df.sort_values(by='Patient_ID',inplace=True)
dwi_t1_all_df.head()

- sort paths

In [ ]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_first_try'

source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in dwi_t1_all_df['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]

print(target_nii_sequences[0])

- convert all files

In [ ]:
#process_directories(source_dcm_sequences, target_nii_sequences)

### Second try with dicom_tags on image for dwi and t1 from manual selection

In [ ]:
dwi_t1_df_2=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_2.csv"))
dwi_t1_df_2.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_2.head()

In [ ]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_second_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors.log')

source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in dwi_t1_df_2['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[1])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]

print(target_nii_sequences[0])

In [ ]:
#process_directories(source_dcm_sequences, target_nii_sequences,log_path=LOG_PATH)# ~ 11 mins

# Try with own small selection

In [ ]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors_pascal_selection.log')

dwi_t1_df_pascal_test=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_pascal_test.csv"))
dwi_t1_df_pascal_test.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_pascal_test.head()

In [ ]:

source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in dwi_t1_df_pascal_test['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[1])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]

print(target_nii_sequences[0])

In [ ]:
# process_directories(source_dcm_sequences, target_nii_sequences,log_path=LOG_PATH)# ~ 11 mins

## Third try with selection from hakim/ Julian / martin  via browsertool 

In [ ]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_third_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors_3.log')

In [ ]:
dwi_t1_df_3=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_3.csv"))
dwi_t1_df_3.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_3.head()

In [ ]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_third_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors_3.log')

source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in dwi_t1_df_2['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]

print(target_nii_sequences[0])

In [ ]:
#process_directories(source_dcm_sequences, target_nii_sequences,log_path=LOG_PATH)# ~ 11 mins